In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
# 暫時禁用future warningq
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
# 專案根目錄（alphagen-master/alphagen-master）
# Jupyter cwd 在 testing/ 時用 parent；否則用 cwd
_cwd = Path.cwd()
root = _cwd.parent if _cwd.name == "testing" else _cwd
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"Project root: {root}")

In [ ]:
#from src.alphas.alpha101 import Alpha101
from src.alphas.alpha191 import Alpha191
alphas  = Alpha191.get_all_factors()
out_dir = '/home/newt/quant_project/src/factor/test_result/alpha101'
# alpha_str = Alpha101.alpha013()  


In [ ]:
import os
from src.factor_engine import FactorEngine

# Alpha101 會用到 $industry / $subindustry；預設 feature_config 把它們排除在表達式外
# （檔案在 ramdisk 也沒用，parser 直接不認）。用專案設定覆寫，並強制重建 engine。
os.environ["ALPHAGEN_FEATURE_CONFIG"] = (
    "/home/newt/quant_project/setting_files/feature_config_alpha101.json"
)
engine = None
df_returns = None

RAM_PATH = "/dev/shm/qlib_ram"

engine = FactorEngine(
    qlib_data_path=RAM_PATH,
    instrument="all",
    start_time="2020-01-01",
    end_time="2026-05-31",
    device="cuda:0",
    max_backtrack_days=300,
    max_future_days=0,
    adjust_factor="back",
)

df_close = engine.calc_df("$close", col_name="close").sort_index()

# (名稱, 賣出日相對 T+1 的偏移)，例如 1day_return = close[T+2]/close[T+1]-1
return_windows = [
    ("1day_return", 2, 1),
    ("3day_return", 4, 1),
    ("5day_return", 6, 1),
    ("10day_return", 11, 1),
    ("20day_return", 21, 1),
    ("30day_return", 31, 1),
]

out = []
for name, end_lead, start_lead in return_windows:
    s = (
        df_close.groupby(level="instrument", group_keys=False)["close"]
        .transform(lambda x: x.shift(-end_lead) / x.shift(-start_lead) - 1)
    )
    s.name = name
    out.append(s)

df_returns = pd.concat(out, axis=1)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def process_untradable(
    df: pd.DataFrame,
    drop: bool = False,
    keep_flag: bool = False,
    show_daily_na: bool = True,
) -> pd.DataFrame:
    """
    篩掉 untradable==1，再可選顯示每日 NA 比例，最後回傳。
    """
    if "$untradable" not in df.columns:
        raise ValueError("DataFrame 中缺少 $untradable 欄位")

    valid = df["$untradable"] == 0
    factor_cols = [c for c in df.columns if c != "$untradable"]
    merged_masked = df.loc[valid].copy()
    if show_daily_na:

        if isinstance(merged_masked.index, pd.MultiIndex):
            dates = merged_masked.index.get_level_values(0)
        elif "date" in merged_masked.columns:
            dates = merged_masked["date"]
        else:
            raise ValueError("df 需有 date 欄位或 (date, instrument) MultiIndex")
        rows = []
        for date, g in merged_masked.groupby(dates):
            n = len(g)
            row = {"date": date, "總筆數": n}
            for col in factor_cols:
                if col not in g.columns:
                    continue
                na_count = g[col].isna().sum()
                pct = (na_count / n * 100) if n else 0.0
                row[f"{col}_NA數"] = na_count
                row[f"{col}_NA_ratio"] = pct
            rows.append(row)
        stats = pd.DataFrame(rows)

        # 用比例做分箱，Y 軸為次數（有幾天落在該比例區間）
        bins = np.linspace(0, 100, 21)  # 0, 5, 10, ..., 100
        n_factors = len([c for c in factor_cols if f"{c}_NA_ratio" in stats.columns])
        fig, axes = plt.subplots(n_factors, 1, figsize=(8, 4 * n_factors))
        if n_factors == 1:
            axes = [axes]
        for ax, col in enumerate(factor_cols):
            if f"{col}_NA_ratio" not in stats.columns:
                continue
            ratio = stats[f"{col}_NA_ratio"]
            counts, bin_edges = np.histogram(ratio, bins=bins)
            bin_labels = [f"{bin_edges[i]:.0f}–{bin_edges[i+1]:.0f}%" for i in range(len(bin_edges)-1)]
            axes[ax].bar(bin_labels, counts, width=0.8, edgecolor="gray")
            axes[ax].set_xlabel("NA ratio")
            axes[ax].set_ylabel("Times")
            axes[ax].set_title(f"{col} daily NA ratio dist")
            axes[ax].tick_params(axis="x", rotation=45)
        plt.tight_layout()
        plt.show()

    if "Alpha" in merged_masked.columns and drop:
        merged_masked = merged_masked.dropna(subset=["Alpha"])

    if keep_flag:
        return merged_masked
    return merged_masked[factor_cols]
def get_tradable_mask(df: pd.DataFrame) -> pd.Series:
    """
    回傳「可交易」布林 mask：$untradable == 0（與 df 同 index）。
    """
    if "$untradable" not in df.columns:
        raise ValueError("DataFrame 中缺少 $untradable 欄位")
    return df["$untradable"] == 0
def apply_tradable_mask(df, mask):
    m = mask.reindex(df.index, fill_value=False)
    return df.loc[m]
df_untradable = engine.load_raw_df(["$untradable"])

tradable_mask = get_tradable_mask(df_untradable)

In [ ]:
import numpy as np
import pandas as pd
from panda_factor.panda_factor.panda_factor.analysis.factor_func import grouping_factor, cal_pct_lag , ext_out_3std , ext_out_mad
from panda_factor.panda_factor.panda_factor.analysis.factor import factor
import base64
import io
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib
import os
import matplotlib.pyplot as plt



from src.alphas.alpha191 import Alpha191
alphas  = Alpha191.get_all_factors()
out_dir = '/home/newt/quant_project/src/factor/test_result/alpha191'



def fig_to_base64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight")
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("utf-8")

def df_to_plotly_table(df, title):
    return go.Figure(data=[go.Table(
        header=dict(values=[""] + list(df.columns), fill_color="lightblue"),
        cells=dict(
            values=[df.index.tolist()] + [df[c].astype(str).tolist() for c in df.columns],
            fill_color="white",
        ),
    )], layout=dict(title=title, height=200 + 25 * len(df)))

# ── 1) 全部 alpha 一次算完（cuda:0）──────────────────────────
# ── 1) 全部 alpha 先算完（cuda:0）；單一公式失敗就跳過，不中斷整批 ──
alpha_names = list(alphas.keys())
alpha_exprs = list(alphas.values())
print(f"批次計算 {len(alpha_names)} 個因子 @ cuda:0 ...")

frames = []
ok_names = []
for name, expr in zip(alpha_names, alpha_exprs):
    try:
        frames.append(engine.calc_df(expr, col_name=name))
        ok_names.append(name)
    except Exception as e:
        print(f"{name}: 計算失敗 — {type(e).__name__}: {e}")

if not frames:
    raise RuntimeError("全部因子計算失敗")

df_all = pd.concat(frames, axis=1)
df_all = apply_tradable_mask(df_all, tradable_mask)
print(f"完成: {len(ok_names)}/{len(alpha_names)} 成功, shape={df_all.shape}")

# ── 2) 逐個拉出來做檢驗 / 篩選 / 報告 ─────────────────────────
factor_name = "Alpha"
for alpha_name in ok_names:
    df = df_all[[alpha_name]].rename(columns={alpha_name: factor_name})
    df = df.dropna(subset=[factor_name])
    df_factor = pd.merge(df, df_returns, on=["datetime", "instrument"], how="left")

    rename_map = {
        "datetime": "date",
        "instrument": "symbol",
    }
    df_factor.reset_index(inplace=True)
    df_factor = df_factor.rename(columns=rename_map)
    df_factor["date"] = pd.to_datetime(df_factor["date"])

    if "returns_lag1" not in df_factor.columns:
        df_factor = cal_pct_lag(df_factor)

    df_factor = df_factor.groupby("date", group_keys=False).apply(
        lambda g: ext_out_3std(g.copy(), factor_name, process_inf=True)
    )
    # grouping_factor 內對因子做 dropna()，當日有 NA 會長度不一致；先只保留有效列
    df_factor_for_grouping = df_factor[np.isfinite(df_factor[factor_name])].copy()
    df_cuted, df_benchmark = grouping_factor(
        df=df_factor_for_grouping,
        factor_name=factor_name,
        group_cnt=5,
    )

    alpha = factor(name=factor_name, group_number=5)
    alpha.set_backtest_parameters(
        period=1,
        predict_direction=1,
    )
    alpha.start_backtest(df_cuted, df_benchmark)

    # IC_mean 或 Rank_IC 任一 > 0.02 且 t統計量 > 2 才畫圖
    ic_info = alpha.df_info2
    ic_mean = float(ic_info.loc["IC_mean", alpha.name])
    rank_ic = float(ic_info.loc["Rank_IC", alpha.name])
    t_stat = float(ic_info.loc["t統計量", alpha.name])
    if not (rank_ic > 0.02 and abs(t_stat) > 2):
        print(f"{alpha_name}: IC_mean={ic_mean}, Rank_IC={rank_ic}, t統計量={t_stat} 未滿足（任一>0.02 且 t>2），略過畫圖")
        continue

    matplotlib.use("Agg")

    _orig_show = plt.show
    plt.show = lambda: None

    alpha.draw_pct()
    img_pct = fig_to_base64(plt.gcf())
    plt.close()
    alpha.draw_ic(0)
    img_ic0 = fig_to_base64(plt.gcf())
    plt.close()
    alpha.draw_ic(1)
    img_ic1 = fig_to_base64(plt.gcf())
    plt.close()
    alpha.draw_ic_dacay()
    img_decay = fig_to_base64(plt.gcf())
    plt.close()
    plt.show = _orig_show

    fig_t1 = df_to_plotly_table(alpha.df_info, f"{alpha_name} 多空分組統計")
    fig_t2 = df_to_plotly_table(alpha.df_info2, f"{alpha_name} IC 統計")

    html_parts = [
        "<!DOCTYPE html><html><head><meta charset='utf-8'></head><body>",
        f"<h1>{alpha_name} 因子報告</h1>",
        fig_t1.to_html(full_html=False, include_plotlyjs="cdn"),
        fig_t2.to_html(full_html=False, include_plotlyjs=False),
        "<h2>分組收益</h2><img src='data:image/png;base64," + img_pct + "' style='max-width:100%'>",
        "<h2>IC 時序 (normal)</h2><img src='data:image/png;base64," + img_ic0 + "' style='max-width:100%'>",
        "<h2>IC 時序 (rank)</h2><img src='data:image/png;base64," + img_ic1 + "' style='max-width:100%'>",
        "<h2>IC 衰減</h2><img src='data:image/png;base64," + img_decay + "' style='max-width:100%'>",
        "</body></html>",
    ]

    html_path = os.path.join(out_dir, f"{alpha_name}_因子報告.html")
    with open(html_path, "w", encoding="utf-8") as f:
        f.write("".join(html_parts))
    print(f"已儲存: {html_path}")
 

In [ ]:
import numpy as np
import pandas as pd
from panda_factor.panda_factor.panda_factor.analysis.factor_func import grouping_factor, cal_pct_lag , ext_out_3std , ext_out_mad
from panda_factor.panda_factor.panda_factor.analysis.factor import factor
import base64
import io
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib
import os
import matplotlib.pyplot as plt



from src.alphas.alpha101 import Alpha101
alphas  = Alpha101.get_all_factors()
out_dir = '/home/newt/quant_project/src/factor/test_result/alpha101'



def fig_to_base64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight")
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("utf-8")

def df_to_plotly_table(df, title):
    return go.Figure(data=[go.Table(
        header=dict(values=[""] + list(df.columns), fill_color="lightblue"),
        cells=dict(
            values=[df.index.tolist()] + [df[c].astype(str).tolist() for c in df.columns],
            fill_color="white",
        ),
    )], layout=dict(title=title, height=200 + 25 * len(df)))

# ── 1) 全部 alpha 一次算完（cuda:0）──────────────────────────
# ── 1) 全部 alpha 先算完（cuda:0）；單一公式失敗就跳過，不中斷整批 ──
alpha_names = list(alphas.keys())
alpha_exprs = list(alphas.values())
print(f"批次計算 {len(alpha_names)} 個因子 @ cuda:0 ...")

frames = []
ok_names = []
for name, expr in zip(alpha_names, alpha_exprs):
    try:
        frames.append(engine.calc_df(expr, col_name=name))
        ok_names.append(name)
    except Exception as e:
        print(f"{name}: 計算失敗 — {type(e).__name__}: {e}")

if not frames:
    raise RuntimeError("全部因子計算失敗")

df_all = pd.concat(frames, axis=1)
df_all = apply_tradable_mask(df_all, tradable_mask)
print(f"完成: {len(ok_names)}/{len(alpha_names)} 成功, shape={df_all.shape}")

# ── 2) 逐個拉出來做檢驗 / 篩選 / 報告 ─────────────────────────
factor_name = "Alpha"
for alpha_name in ok_names:
    df = df_all[[alpha_name]].rename(columns={alpha_name: factor_name})
    df = df.dropna(subset=[factor_name])
    df_factor = pd.merge(df, df_returns, on=["datetime", "instrument"], how="left")

    rename_map = {
        "datetime": "date",
        "instrument": "symbol",
    }
    df_factor.reset_index(inplace=True)
    df_factor = df_factor.rename(columns=rename_map)
    df_factor["date"] = pd.to_datetime(df_factor["date"])

    if "returns_lag1" not in df_factor.columns:
        df_factor = cal_pct_lag(df_factor)

    df_factor = df_factor.groupby("date", group_keys=False).apply(
        lambda g: ext_out_3std(g.copy(), factor_name, process_inf=True)
    )
    # grouping_factor 內對因子做 dropna()，當日有 NA 會長度不一致；先只保留有效列
    df_factor_for_grouping = df_factor[np.isfinite(df_factor[factor_name])].copy()
    df_cuted, df_benchmark = grouping_factor(
        df=df_factor_for_grouping,
        factor_name=factor_name,
        group_cnt=5,
    )

    alpha = factor(name=factor_name, group_number=5)
    alpha.set_backtest_parameters(
        period=1,
        predict_direction=1,
    )
    alpha.start_backtest(df_cuted, df_benchmark)

    # IC_mean 或 Rank_IC 任一 > 0.02 且 t統計量 > 2 才畫圖
    ic_info = alpha.df_info2
    ic_mean = float(ic_info.loc["IC_mean", alpha.name])
    rank_ic = float(ic_info.loc["Rank_IC", alpha.name])
    t_stat = float(ic_info.loc["t統計量", alpha.name])
    if not (rank_ic > 0.02 and abs(t_stat) > 2):
        print(f"{alpha_name}: IC_mean={ic_mean}, Rank_IC={rank_ic}, t統計量={t_stat} 未滿足（任一>0.02 且 t>2），略過畫圖")
        continue

    matplotlib.use("Agg")

    _orig_show = plt.show
    plt.show = lambda: None

    alpha.draw_pct()
    img_pct = fig_to_base64(plt.gcf())
    plt.close()
    alpha.draw_ic(0)
    img_ic0 = fig_to_base64(plt.gcf())
    plt.close()
    alpha.draw_ic(1)
    img_ic1 = fig_to_base64(plt.gcf())
    plt.close()
    alpha.draw_ic_dacay()
    img_decay = fig_to_base64(plt.gcf())
    plt.close()
    plt.show = _orig_show

    fig_t1 = df_to_plotly_table(alpha.df_info, f"{alpha_name} 多空分組統計")
    fig_t2 = df_to_plotly_table(alpha.df_info2, f"{alpha_name} IC 統計")

    html_parts = [
        "<!DOCTYPE html><html><head><meta charset='utf-8'></head><body>",
        f"<h1>{alpha_name} 因子報告</h1>",
        fig_t1.to_html(full_html=False, include_plotlyjs="cdn"),
        fig_t2.to_html(full_html=False, include_plotlyjs=False),
        "<h2>分組收益</h2><img src='data:image/png;base64," + img_pct + "' style='max-width:100%'>",
        "<h2>IC 時序 (normal)</h2><img src='data:image/png;base64," + img_ic0 + "' style='max-width:100%'>",
        "<h2>IC 時序 (rank)</h2><img src='data:image/png;base64," + img_ic1 + "' style='max-width:100%'>",
        "<h2>IC 衰減</h2><img src='data:image/png;base64," + img_decay + "' style='max-width:100%'>",
        "</body></html>",
    ]

    html_path = os.path.join(out_dir, f"{alpha_name}_因子報告.html")
    with open(html_path, "w", encoding="utf-8") as f:
        f.write("".join(html_parts))
    print(f"已儲存: {html_path}")
 

In [ ]:
# 指定要抓的 alpha（改這裡）
ALPHA = "alpha012"
SYMBOL = None  # 例如 "2330"；None = 全部股票

# 優先用批次結果 df_all；沒算過或沒這欄就單獨算
if "df_all" in globals() and ALPHA in getattr(df_all, "columns", []):
    s = df_all[ALPHA]
else:
    if ALPHA not in alphas:
        raise KeyError(f"{ALPHA} 不在 alphas，可用: {list(alphas)[:5]} ...")
    s = engine.calc_df(alphas[ALPHA], col_name=ALPHA)[ALPHA]
    if "tradable_mask" in globals():
        s = s.loc[tradable_mask.reindex(s.index, fill_value=False)]

df_alpha = (
    s.rename("value")
    .reset_index()
    .rename(columns={"datetime": "date", "instrument": "symbol"})
)
if SYMBOL is not None:
    df_alpha = df_alpha.loc[df_alpha["symbol"].astype(str) == str(SYMBOL)]

df_alpha = df_alpha[["date", "symbol", "value"]].sort_values(["date", "symbol"]).reset_index(drop=True)
print(f"{ALPHA}: rows={len(df_alpha):,}  date=[{df_alpha['date'].min()} ~ {df_alpha['date'].max()}]")
df_alpha